# The decisions inside an enzyme-kinetics fit

This notebook walks through one complete analysis — Michaelis–Menten parameters, the
comparison with three linearisations, bootstrap intervals, inhibition typing and a
calibration with a detection limit — on the simulated datasets committed in
`datasets/sample`.

It is here to be read in order and run top to bottom. Every number it prints comes from
the data files, and every caveat it prints comes from the library rather than from this
notebook, so a fit that is not trustworthy says so by itself.

**All data are simulated.** True parameters are recorded in `datasets/metadata/*.yml`, so
you can check the estimates against the truth as you go — a luxury you never have with
real data, and the reason simulation is used for teaching.


## 1. Setup

In [1]:
import sys
import numpy as np
import kinetics
from kinetics.fitting import (fit_michaelis_menten, compare_linearisations,
                              bootstrap_ci)
from kinetics.inhibition import compare_inhibition_models
from kinetics.calibration import fit_calibration
from kinetics.report import results_to_markdown

print("kinetics", kinetics.__version__, "| numpy", np.__version__, "| python", sys.version.split()[0])
print("units: [S], [I] in µM; v in µmol min^-1 (mg protein)^-1; no unit conversion anywhere")

from pathlib import Path

# The notebook must not assume a working directory: nbconvert runs it with the
# notebook's own directory as cwd, while scripts/check_notebooks.py runs it from the
# repository root. Walk up until the data are where they should be.
ROOT = Path.cwd()
while not (ROOT / "datasets").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("repository root:", ROOT)


kinetics 0.1.0 | numpy 2.3.5 | python 3.13.16
units: [S], [I] in µM; v in µmol min^-1 (mg protein)^-1; no unit conversion anywhere
repository root: /home/user/repos/python-for-biochemists


## 2. Read the data without a library

The file has three replicates at each substrate concentration. Reading it with the
standard library first is deliberate: a notebook that starts with `pd.read_csv` hides the
question of what the rows mean.


In [2]:
import csv
from pathlib import Path

path = ROOT / "datasets/sample/michaelis_menten.csv"
rows = list(csv.DictReader(path.open(encoding="utf-8")))
print(f"{path} -> {len(rows)} rows, columns {list(rows[0])}")
for row in rows[:4]:
    print(row)


/home/user/repos/python-for-biochemists/datasets/sample/michaelis_menten.csv -> 27 rows, columns ['S', 'replicate', 'v']
{'S': '1.0', 'replicate': '1', 'v': '0.33515'}
{'S': '1.0', 'replicate': '2', 'v': '0.37198'}
{'S': '1.0', 'replicate': '3', 'v': '0.37316'}
{'S': '2.5', 'replicate': '1', 'v': '0.96812'}


## 3. Fit Michaelis–Menten, and notice what the fit does not tell you

`Vmax` is an extrapolation to infinite substrate concentration. The fit returns an
estimate with a standard error as if the model were correct — the standard error does not
know whether the mechanism is right.


In [3]:
S = np.array([float(r["S"]) for r in rows])
v = np.array([float(r["v"]) for r in rows])

fit = fit_michaelis_menten(S, v)
print(f"Vmax = {fit.params['vmax']:.3f} ± {fit.stderr['vmax']:.3f}")
print(f"Km   = {fit.params['km']:.3f} ± {fit.stderr['km']:.3f} µM")
print(f"R²   = {fit.r_squared:.5f}   AICc = {fit.aicc:.2f}   n = {fit.n}")
print()
for note in fit.notes:
    print("note:", note)


Vmax = 9.984 ± 0.107
Km   = 24.877 ± 0.840 µM
R²   = 0.99742   AICc = -97.06   n = 27

note: Vmax is an extrapolation to infinite [S] and is never measured directly


Compare with the truth recorded when the data were generated: `Vmax = 10.0`,
`Km = 25.0` µM. The estimate is close — because the error model here happens to match
the assumptions of ordinary least squares reasonably well.


## 4. The linearisations: same data, four answers

`compare_linearisations` fits the same points four ways. The linearised methods transform
the data and then run ordinary least squares on the transformed scale, which changes what
"best fit" means.


In [4]:
results = compare_linearisations(S, v)
print(f"{'method':<18}{'Vmax':>9}{'Km':>9}  bias note")
for result in results:
    print(f"{result.model:<18}{result.params['vmax']:>9.3f}{result.params['km']:>9.3f}  "
          f"{'linearised' if result.method == 'linearised' else 'nonlinear'}")
print()
print("truth: Vmax 10.0, Km 25.0")


method                 Vmax       Km  bias note
michaelis_menten      9.984   24.877  nonlinear
hanes                 9.980   25.012  linearised
eadie_hofstee         9.987   25.043  linearised
lineweaver_burk      12.045   32.140  linearised

truth: Vmax 10.0, Km 25.0


The spread is small here because this dataset has nine substrate concentrations and 3 %
noise. `tests/test_linearisation.py` and `figures/fig2_linearisation_bias.png` show what
happens over 400 simulated assays: the Lineweaver–Burk estimate is not merely noisier, it
is heavy-tailed, and only 57 % of estimates land within 20 % of the truth.


## 5. Bootstrap intervals: what would move if you repeated the experiment

In [5]:
ci = bootstrap_ci(S, v, n_resamples=2000, seed=20261107)
print("Vmax 95 % percentile bootstrap:", tuple(round(x, 3) for x in ci["vmax"]))
print("Km   95 % percentile bootstrap:", tuple(round(x, 3) for x in ci["km"]))
print()
print("The seed is part of the result. Without it the interval cannot be reproduced,")
print("and an interval you cannot reproduce is not a result.")


Vmax 95 % percentile bootstrap: (9.71, 10.263)
Km   95 % percentile bootstrap: (23.359, 26.449)

The seed is part of the result. Without it the interval cannot be reproduced,
and an interval you cannot reproduce is not a result.


## 6. Typing an inhibition mechanism, and failing to

Two datasets, same generator, different experimental design. In the first, the inhibitor
concentrations span 0–2·Ki and the mechanism is identifiable. In the second, `[I]` never
exceeds 0.004·Ki and no amount of fitting can recover the mechanism.


In [6]:
def load_inhibition(name):
    return [{"S": float(r["S"]), "I": float(r["I"]), "v": float(r["v"])}
            for r in csv.DictReader(open(ROOT / f"datasets/sample/{name}.csv", encoding="utf-8"))]

for name in ("inhibition_series", "inhibition_weak"):
    comparison = compare_inhibition_models(load_inhibition(name))
    deltas = ", ".join(f"{m} Δ={d:.2f}" for m, d in sorted(comparison.delta_aicc.items()))
    print(f"{name}: best = {comparison.best}   [{deltas}]")
    print(f"  truth recorded in the metadata: "
          f"{__import__('yaml').safe_load(open(ROOT / f'datasets/metadata/{name}.yml'))['true_parameters']['mechanism']}")
    ambiguous = [n for n in comparison.notes if "indistinguishable" in n]
    print("  ", ambiguous[0] if ambiguous else "no ambiguity reported")
    print()


inhibition_series: best = competitive   [competitive Δ=0.00, noncompetitive Δ=42.55, uncompetitive Δ=57.90]
  truth recorded in the metadata: competitive
   no ambiguity reported

inhibition_weak: best = uncompetitive   [competitive Δ=1.68, noncompetitive Δ=0.48, uncompetitive Δ=0.00]
  truth recorded in the metadata: competitive
   indistinguishable models (ΔAICc ≤ 2): competitive, noncompetitive — report the ambiguity rather than the winner



The second line of each block is the point of the exercise: the nominal winner is **not**
the mechanism that generated the data, and the library says so rather than presenting a
confident answer to an underpowered experiment.


## 7. A calibration, two ways, and a detection limit that changes by 58×

The standards have proportional error (about 4 %), so the unweighted fit is dominated by
the noisiest standards.


In [7]:
cal = list(csv.DictReader(open(ROOT / "datasets/sample/calibration_mean.csv", encoding="utf-8")))
x = np.array([float(r["concentration"]) for r in cal])
y = np.array([float(r["signal"]) for r in cal])
sd = np.array([float(r["sd"]) for r in cal])

for label, kwargs in (("unweighted", {}), ("weighted", {"weighted": True, "replicate_sd": sd})):
    result = fit_calibration(x, y, **kwargs)
    print(f"{label:<11} slope {result.slope:.5f}  intercept {result.intercept:.5f}  "
          f"LOD {result.lod:7.3f} µM  LOQ {result.loq:7.3f} µM")
print()
print("truth: slope 0.0485, intercept 0.011 (0.0025 absorbance units of blank noise)")


unweighted  slope 0.04764  intercept 0.01245  LOD   6.358 µM  LOQ  19.266 µM
weighted    slope 0.04901  intercept 0.01234  LOD   0.109 µM  LOQ   0.331 µM

truth: slope 0.0485, intercept 0.011 (0.0025 absorbance units of blank noise)


## 8. A report you can hand over

The same numbers, rendered from the same objects. Nothing in the report is retyped.


In [8]:
print(results_to_markdown(results, title="Michaelis-Menten and the linearisations",
                          context={"dataset": str(path), "units": "µM; µmol min⁻¹ mg⁻¹",
                                   "bootstrap": "2000 resamples, seed 20261107"}))


# Michaelis-Menten and the linearisations

| field | value |
|---|---|
| dataset | /home/user/repos/python-for-biochemists/datasets/sample/michaelis_menten.csv |
| units | µM; µmol min⁻¹ mg⁻¹ |
| bootstrap | 2000 resamples, seed 20261107 |

## Model comparison

| model | method | n | k | R² | AICc |
|---|---|---:|---:|---:|---:|
| michaelis_menten | nonlinear | 27 | 2 | 0.99742 | -97.06 |
| hanes | linearised | 27 | 2 | 0.99815 | not comparable |
| eadie_hofstee | linearised | 27 | 2 | 0.98398 | not comparable |
| lineweaver_burk | linearised | 27 | 2 | 0.99521 | not comparable |

## Parameters

| model | parameter | estimate | std. error |
|---|---|---:|---:|
| michaelis_menten | vmax | 9.98381 | 0.107 |
| michaelis_menten | km | 24.8774 | 0.84 |
| hanes | slope | 0.100203 | 0.000863 |
| hanes | intercept | 2.50631 | — |
| hanes | vmax | 9.97976 | — |
| hanes | km | 25.0124 | — |
| eadie_hofstee | slope | -25.0426 | 0.639 |
| eadie_hofstee | intercept | 9.98669 | — |
| eadie_hofstee |

## 9. What this notebook did not do

* No unit conversion: the units are whatever the file used.
* No model selection beyond AICc: a lower AICc is evidence, not proof, and it cannot
  rule out a mechanism that was never a candidate.
* No measurement: every dataset in this repository is simulated, and each one says so in
  `datasets/metadata/`.
